# D_Gold - Modelo estrella

Gold consume las tablas Silver registradas en Unity Catalog y construye
4 dimensiones y 1 tabla de hechos, listas para Power BI.


In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

sales = spark.table(f"{SILVER_SCHEMA}.sales")


def write_gold(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

df_dim_products = spark.table(f"{GOLD_SCHEMA}.dim_products")
df_dim_stores = spark.table(f"{GOLD_SCHEMA}.dim_stores")
df_dim_customers = spark.table(f"{GOLD_SCHEMA}.dim_customers")
df_dim_calendar = spark.table(f"{GOLD_SCHEMA}.dim_calendar")


# Tabla de hechos
df_fact_sales = (
    sales.alias("s")
    .join(df_dim_products.alias("p"), F.col("s.product_id") == F.col("p.product_id"), "left")
    .join(df_dim_stores.alias("st"), F.col("s.store_id") == F.col("st.store_id"), "left")
    .join(df_dim_customers.alias("c"), F.col("s.customer_id") == F.col("c.customer_id"), "left")
    .join(df_dim_calendar.alias("cal"), F.col("s.order_date") == F.col("cal.date"), "left")
    .select(
        F.col("s.order_id"),
        F.col("s.order_date"),
        F.col("cal.date_key"),
        F.col("p.product_id"),
        F.col("st.store_id"),
        F.col("c.customer_id"),
        F.col("s.quantity"),
        F.col("s.unit_price"),
        F.col("s.discount"),
        F.col("s.revenue"),
        F.col("s.cost"),
        F.col("s.profit"),
    )
)
write_gold(df_fact_sales, f"{GOLD_SCHEMA}.fact_sales")

print("GOLD - modelo estrella generado correctamente")
print("Hechos: fact_sales")


In [0]:
%sql

SELECT 'fact_sales' AS tabla, COUNT(*) AS total_filas FROM workspace.gold.fact_sales;


In [0]:
%sql
select * from workspace.gold.fact_sales limit 3